In [ ]:
using CUDA
using LinearAlgebra
using Random
using LatticeDiracOperators
using InteractiveUtils
using Gaugefields
import Gaugefields.AbstractGaugefields_module:getvalue,setvalue!

println("Test for 1/D for Wilson CUDA")

versioninfo()

version = Base.pkgversion(Gaugefields)
println("version of Gaugefields : ", version)
version = Base.pkgversion(LatticeDiracOperators)
println("version of LatticeDiracOperators: ", version)

const pes =  (4, 4, 4, 4)
const cuda = true
const numtrj_in = 2 # the number of operations to take average of performance
const MDsteps_in = 0 # dummy

const βin = 6.0
const NXin = 8
const NYin = 8
const NZin = 8
const NTin = 16
const κin = 0.141139

header = """
Parameters
const blocks = $pes
const cuda = $cuda
const numtrj_in = $numtrj_in
const MDsteps_in = $MDsteps_in

const βin = $βin
const NXin = $NXin
const NYin = $NYin
const NZin = $NZin
const NTin = $NTin
const κin = $κin
"""

function test1()
    NX = NXin
    NY = NYin
    NZ = NZin
    NT = NTin
    Nwing = 0
    Dim = 4
    NC = 3

    if cuda
        PEs = pes#(1,1,1,2)
        U = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold",cuda=true,blocks=PEs)

    else
        U = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold")
    end
    Ucpu = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold")
    substitute_U!(Ucpu,U)

    Random.seed!(123+get_myrank(U[1]))

    if get_myrank(U) == 0
        println(header) # print header
    end

    snet = GaugeAction(U)
    plaqloop = make_loops_fromname("plaquette")
    append!(plaqloop,plaqloop')
    β = βin/2
    push!(snet,β,plaqloop)

    #show(snet)

    Random.seed!(123)
    x = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
    xcpu = Initialize_pseudofermion_fields(Ucpu[1],"Wilson",nowing=true)

    params = Dict()
    params["Dirac_operator"] = "Wilson"
    params["κ"] = κin
    params["faster version"] = true
    params["eps_CG"] = 1.0e-16
    params["verbose_level"] = 2 # to show CG counts

    D = Dirac_operator(U,x,params)
    Dcpu = Dirac_operator(Ucpu,xcpu,params)
    gauss_distribution_fermion!(xcpu)

    substitute_fermion!(x,xcpu)


    b = similar(x)
    y = similar(x)
    bcpu = similar(xcpu)
    ycpu = similar(xcpu)
    mul!(b,D,x)
    @time for i in 1:100
     mul!(b,D,x)
    end 
    mul!(bcpu,Dcpu,xcpu)
    @time for i in 1:100
     mul!(bcpu,Dcpu,xcpu)
    end



    println_verbose_level1(U[1],"Benchmark!!")
    @time solve_DinvX!(y,D,b)
    @time solve_DinvX!(ycpu,Dcpu,bcpu)



    return
end
test1()

Test for 1/D for Wilson CUDA


Julia Version 1.10.7
Commit 4976d05258 (2024-11-26 15:57 UTC)
Build Info:
  Official https://julialang.org/ release
Platform Info:
  OS: Windows (x86_64-w64-mingw32)
  CPU: 32 × AMD Ryzen 9 7950X3D 16-Core Processor          
  WORD_SIZE: 64
  LIBM: libopenlibm
  LLVM: libLLVM-15.0.7 (ORCJIT, znver3)
Threads: 1 default, 0 interactive, 1 GC (on 32 virtual cores)
version of Gaugefields : 0.5.15
version of LatticeDiracOperators: 0.4.14
Parameters
const blocks = (4, 4, 4, 4)
const cuda = true
const numtrj_in = 2
const MDsteps_in = 0

const βin = 6.0
const NXin = 8
const NYin = 8
const NZin = 8
const NTin = 16
const κin = 0.141139

  0.382088 seconds (83.63 k allocations: 3.402 MiB)
  0.326243 seconds (2.80 k allocations: 281.250 KiB)
Benchmark!!
  2.932729 seconds (4.26 M allocations: 293.882 MiB, 2.01% gc time, 50.78% compilation time)
  1.751951 seconds (597.03 k allocations: 39.987 MiB, 55.30% compilation time)


: 